# Assignment 4: Recurrent Neural Networks (41 marks total)
### Due: November 19 at 11:59pm (grace period until November 21 at 11:59pm)

### Name:

The goal of this assignment is to apply Recurrent Neural Networks (RNNs) in PyTorch for text data classification.

## Part 1: LSTM

### Step 0: Import Libraries

In [43]:
import torch
from datasets import load_dataset
from collections import Counter
import re
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

In [44]:
import warnings
warnings.filterwarnings(action='ignore')

### Step 1: Data Loading and Preprocessing (12 marks)

For this assignment, we will be using the imdb dataset from the 🤗 Datasets library

In [45]:
# TO DO: Load the dataset (1 mark)
Imdb_data = load_dataset("imdb")
train_data = Imdb_data["train"]
test_data = Imdb_data["test"]





We need to preprocess the data before we can feed it into the model. The first step is to define a custom tokenizer to perform the following tasks: 
- Extract the text data from the dataset
- Remove any non-alphanumeric characters
- Separate each data sample into separate words (tokens)

In [46]:
def tokenizer(data_iter):
    '''Tokenizes the input data
    input: data_iter (type: dictionary)
    output: text (type: list[list[str]])
    '''
    # TO DO: fill in this function (2 marks)
    text = data_iter['text']
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    return [tokens]


We will also need to extract the labels from the dataset. Complete the label_extractor function below:

In [47]:
def label_extractor(data_iter):
    '''Takes the label for each data sample and stores it in a separate list
    input: data_iter (type: dictionary)
    output: labels (type: list)
    '''
    # TO DO: fill in this function (1 mark)
    return [data_iter['label']]


Now that we have the text data separated into words, we need to define the vocabulary. We cannot keep all the words in the vocabulary, so we want to limit the vocabulary size and only take the most common words. In this case, the maximum vocabulary size is 10,000 words. Any word that is excluded will be set to an unknown token. You can use the function below to build the vocabulary:

In [48]:
# Build a vocabulary
def build_vocab(data_iter, max_size=10000):
    '''Creates a vocabulary based on the training data
    input: data_iter (type: list[list[str]])
    output: vocab (type: dictionary)
    '''
    counter = Counter()
    for words in data_iter:
        counter.update(words)
    # Filter to most common words
    vocab = {word: i + 1 for i, (word, _) in enumerate(counter.most_common(max_size))}
    # Add a token for unknown words (0)
    vocab['<unk>'] = 0 
    return vocab

In the vocabulary, each word is mapped to a number in the vocabulary. We will need to encode the dataset based on these numbers, as tensors cannot handle string data.

The next step is to pad or truncate each sequence based on a maximum length, to make sure that the dataset can be transformed into a tensor (as discussed in class).

Fill in the function below to encode and pad the dataset:

In [49]:
def encode_and_pad(text, vocab, max_len=100):
    '''Encode and pad the input text dataset
    input: text (type: list[list[str]])
    input: vocab (type: dictionary)
    input: max_len (type: int)
    output: texts (type: list[list[str]])
    '''
    # TO DO: fill in the function to encode text to integers and pad/truncate sequences (2 marks)
    encoded_texts = []
    padding_index = 0
    unk_index = vocab.get('<unk>', 0)

    for tokens in text:
        encoded = [vocab.get(token, unk_index) for token in tokens]
        encoded = encoded[:max_len]
        if len(encoded) < max_len:
            encoded += [padding_index] * (max_len - len(encoded))
        encoded_texts.append(encoded)

    return encoded_texts


The next step is to create a custom PyTorch Dataset class that calls the `encode_and_pad()` function and stores the text and labels as tensors. Fill in the `init` portion of the class: 

In [50]:
# Create a custom PyTorch Dataset class
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        # TO DO: call the encode_and_pad() function and set self.texts and self.labels (2 marks)
        en_texts = encode_and_pad(texts, vocab, max_len)

        self.texts = torch.tensor(en_texts, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.float32)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]


Now you can call all the functions that have been created:

In [51]:
MAX_LEN = 256  # Sequence length
BATCH_SIZE = 64

# TO DO: Tokenize training data (1 mark)
train_text = [tokenizer(sample)[0] for sample in train_data]
test_text = [tokenizer(sample)[0] for sample in test_data]

# TO DO: Extract labels from training and testing data (1 mark)
train_labels = [label_extractor(sample)[0] for sample in train_data]
test_labels = [label_extractor(sample)[0] for sample in test_data]

# TO DO: Build Vocabulary (from training data only) (1 mark)
vocab = build_vocab(train_text)

# TO DO: Prepare datasets (using TextDataset class) and store datasets using DataLoaders (1 mark)
train_dataset = TextDataset(train_text, train_labels, vocab, MAX_LEN)
test_dataset = TextDataset(test_text, test_labels, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


### Step 2: Define Model (4 marks)

For this assignment, we will be using the LSTM model. Inside the LSTM model, the first layer will be an embedding layer, to convert the singular numerical representation of each word into an embedded vector. We can use `nn.Embedding(...)` for this.

Define LSTMClassifier below:

In [52]:
# TO DO: Define LSTM class (4 marks)
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers=1):
        super().__init__()
        # TO DO: Embedding layer 
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        # TO DO: LSTM layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        # TO DO: Linear fully-connected layer
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # TO DO: Fill in the model steps
        # NOTE: The LSTM outputs (output, (hidden, cell)) - hidden and cell are not used
        # NOTE: Use the hidden state from the final time step for the fc layer
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        final_hidden = hidden[-1]
        output = self.fc(final_hidden)
        return output


### Step 3: Define Training and Testing Loops (4 marks)

The next step is to define functions for the training and testing loops. For this case, we will only be calculating the loss at each epoch.

In [53]:
# TO DO: Define training loop (2 marks)
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    ttl_correct = 0
    ttl_samples = 0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.float().to(device)

        optimizer.zero_grad()
        outputs = model(inputs).squeeze(1)

        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

   
        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).float()

        ttl_correct += (preds == targets).sum().item()
        ttl_samples += targets.size(0)

    avg_loss = running_loss / len(dataloader.dataset)
    accuracy = 100 * ttl_correct / ttl_samples

    return avg_loss, accuracy


In [54]:
# TO DO: Define testing loop (2 marks)
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    ttl_correct = 0
    ttl_samples = 0

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(device)
            targets = targets.float().to(device)

            outputs = model(inputs).squeeze(1)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * inputs.size(0)


            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()

            ttl_correct += (preds == targets).sum().item()
            ttl_samples += targets.size(0)

    avg_loss = running_loss / len(dataloader.dataset)
    accuracy = 100 * ttl_correct / ttl_samples

    return avg_loss, accuracy


### Step 4: Train and Evaluate (3 marks)

Now that we have all the necessary functions, we can select our hyperparameters, and train and evaluate our model. For this case, since we are not comparing different models, we do not need a validation set.

In [55]:
# Hyperparameters
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = 1 # Binary classification
NUM_LAYERS = 1

In [56]:
# TO DO: Create model object (1 mark)
model = LSTMClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, NUM_LAYERS)


In [57]:
import torch.optim as optim

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

LSTMClassifier(
  (embedding): Embedding(10001, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

Since this case is binary optimization, we will use the binary cross entropy criterion, `BCEWithLogitsLoss()`. This model is similar to Cross Entropy, but uses a sigmoid layer instead of a softmax layer. For the optimization function, we will use Adam with a learning rate of 0.01.

In [58]:
# TO DO: Define optimization model and criterion (1 mark)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


We can now run our training and testing loops. Since this takes a long time to run, we will set the number of epochs to 5. Print out the training and testing losses.

In [60]:
# TO DO: Run training and testing loops and print losses for each epoch (1 mark)
EPOCHS = 5
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch + 1}/{EPOCHS} - "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%  "
          f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")


Epoch 1/5 - Train Loss: 0.6722, Train Acc: 57.79%  Test Loss: 0.6863, Test Acc: 55.54%
Epoch 2/5 - Train Loss: 0.6140, Train Acc: 66.53%  Test Loss: 0.6343, Test Acc: 65.18%
Epoch 3/5 - Train Loss: 0.5443, Train Acc: 73.00%  Test Loss: 0.5985, Test Acc: 69.29%
Epoch 4/5 - Train Loss: 0.4508, Train Acc: 79.62%  Test Loss: 0.5333, Test Acc: 74.26%
Epoch 5/5 - Train Loss: 0.4004, Train Acc: 82.36%  Test Loss: 0.5140, Test Acc: 77.05%


## Part 2: Questions and Process Description

### Questions (12 marks)

1. Do you think this model worked well to classify the data? Why or why not? Can you make a good decision about this only using loss data?
2. What could you do to further improve the results? Provide two suggestions.
3. Why does a simple RNN often underperform compared to LSTM or GRU on long text sequences such as IMDB reviews?
4. Why does the embedding layer improve performance compared to one-hot encoding?
5. If we switched to character-level input instead of word-level, what changes would we expect in performance and training time?
6. How does vocabulary size influence model performance and generalization?


1.training loss goes from 0.67 to 0.4 which means that the training model was learning and improving. but the test loss dint really cvhnage muhc which could mena many thungs like the model has reached its limit on its learning capacity. so yes i do fele that the model worked well but no i do not think that you can make a decision by only looking at the loss data

2.You could try using a bigger model, like increasing the embedding size or the LSTM’s hidden units. Adding something like dropout or weight decay can also help the model generalize better.

3.A basic RNN forgets information quickly because its gradients fade over long sequences. LSTMs and GRUs are built to hold onto important context, so they handle long text much better.

4.Embeddings give each word a meaningful dense vector, so the model can understand similarities between words. One-hot vectors don’t carry any meaning, so they’re much less useful.

5.Training becomes a lot slower because character sequences are much longer than word sequences. The accuracy usually drops too, since individual characters don’t carry as much meaning.

6.If the vocabulary is too small, the model loses important words and becomes less accurate. If it’s too big, the model gets heavier, slower, and more likely to overfit.

### Process Description (4 marks)
Please describe the process you used to create your code. Cite any websites or generative AI tools used. You can use the following questions as guidance:
1. Where did you source your code?
1. In what order did you complete the steps?
1. If you used generative AI, what prompts did you use? Did you need to modify the code at all? Why or why not?
1. Did you have any challenges? If yes, what were they? If not, what helped you to be successful?

1.I mainly used the example files from D2L and used Chat GPT to help fix sectiosn of code or to guide me.

2.I went in the order as was given in the assignment.

3.I used short prompts like “is this correct?”, “how would you go about this,”. Yes i modified it to match the assignment spec.

4.My main challenges were keeping the data formats consistent and making each function match the template. Checking each step carefully made the process easier.

## Part 3: Reflection (2 marks)

Include a sentence or two about:

- what you liked or disliked,
- found interesting, confusing, challenging, motivating
while working on this assignment.

Answer 

- I liked seeing the LSTM actually work on real text instead of just learning the theory, and it was nice to see the loss fall.
- The lower number of epochs made the assignment run faster and overall much smoother to work through.

